In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "twcs.csv"

print("Dataset path:")
print(DATA_PATH)

Dataset path:
c:\Users\arnav\OneDrive\Desktop\hiver-ai-support-agent\data\raw\twcs.csv


In [4]:
df_sample = pd.read_csv(
    DATA_PATH,
    nrows=100_000
)

print("Sample shape:", df_sample.shape)

Sample shape: (100000, 7)


In [5]:
print("Columns:")
for column in df_sample.columns:
    print("-", column)

Columns:
- tweet_id
- author_id
- inbound
- created_at
- text
- response_tweet_id
- in_response_to_tweet_id


In [6]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   tweet_id                 100000 non-null  int64  
 1   author_id                100000 non-null  str    
 2   inbound                  100000 non-null  bool   
 3   created_at               100000 non-null  str    
 4   text                     100000 non-null  str    
 5   response_tweet_id        67571 non-null   str    
 6   in_response_to_tweet_id  74090 non-null   float64
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 20.3 MB


In [7]:
df_sample.describe()

,tweet_id,in_response_to_tweet_id
count,100000.000000,74090.000000
mean,63430.983200,63062.375422
std,36612.198664,36551.446104
min,1.000000,1.000000
25%,31494.750000,31049.250000
50%,61711.500000,61440.000000
75%,96484.250000,95919.500000
max,126124.000000,126125.000000


In [8]:
df_sample[[
    "tweet_id",
    "author_id",
    "inbound",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id"
]].head(10)

,tweet_id,author_id,inbound,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,@sprintcare I did.,4,6.0
5,6,sprintcare,False,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,@115713 We understand your concerns and we'd l...,12,16.0


In [9]:
missing = pd.DataFrame({
    "missing_count": df_sample.isna().sum(),
    "missing_percentage": (
        df_sample.isna().mean() * 100
    ).round(2)
})

missing

,missing_count,missing_percentage
tweet_id,0,0.00
author_id,0,0.00
inbound,0,0.00
created_at,0,0.00
text,0,0.00
response_tweet_id,32429,32.43
in_response_to_tweet_id,25910,25.91


In [10]:
print("Inbound distribution:")
print(df_sample["inbound"].value_counts())

print("\nPercentage:")
print(
    (df_sample["inbound"].value_counts(normalize=True) * 100)
    .round(2)
)

Inbound distribution:
inbound
True     54948
False    45052
Name: count, dtype: int64

Percentage:
inbound
True     54.95
False    45.05
Name: proportion, dtype: float64


In [11]:
print(
    "Unique author IDs:",
    df_sample["author_id"].nunique()
)

Unique author IDs: 27592


In [12]:
author_counts = (
    df_sample["author_id"]
    .value_counts()
    .head(30)
)

author_counts

author_id
AmazonHelp         6924
AppleSupport       3106
Uber_Support       2102
Tesco              1450
SpotifyCares       1356
British_Airways    1156
comcastcares       1092
sainsburys         1081
Delta              1039
VirginTrains       1001
AmericanAir         956
ChipotleTweets      889
O2                  843
GWRHelp             826
hulu_support        823
XboxSupport         816
TMobileHelp         816
SouthwestAir        793
Ask_Spectrum        763
idea_cares          743
ArgosHelpers        697
AirAsiaSupport      658
sprintcare          646
UPSHelp             632
VerizonSupport      631
MicrosoftHelps      622
AskPlayStation      621
Safaricom_Care      591
ATVIAssist          550
SW_Help             532
Name: count, dtype: int64

In [13]:
company_tweets = df_sample[
    df_sample["inbound"] == False
].copy()

print(
    "Company-side tweets:",
    len(company_tweets)
)

print(
    "Company-side accounts:",
    company_tweets["author_id"].nunique()
)

Company-side tweets: 45052
Company-side accounts: 105


In [14]:
company_activity = (
    company_tweets["author_id"]
    .value_counts()
    .rename("company_replies")
)

company_activity.head(30)

author_id
AmazonHelp         6924
AppleSupport       3106
Uber_Support       2102
Tesco              1450
SpotifyCares       1356
British_Airways    1156
comcastcares       1092
sainsburys         1081
Delta              1039
VirginTrains       1001
AmericanAir         956
ChipotleTweets      889
O2                  843
GWRHelp             826
hulu_support        823
XboxSupport         816
TMobileHelp         816
SouthwestAir        793
Ask_Spectrum        763
idea_cares          743
ArgosHelpers        697
AirAsiaSupport      658
sprintcare          646
UPSHelp             632
VerizonSupport      631
MicrosoftHelps      622
AskPlayStation      621
Safaricom_Care      591
ATVIAssist          550
SW_Help             532
Name: company_replies, dtype: int64

In [15]:
customer_tweets = df_sample[
    df_sample["inbound"] == True
].copy()

customer_activity = (
    customer_tweets["author_id"]
    .value_counts()
    .rename("customer_tweets")
)

customer_activity.head(30)

author_id
117242    56
116230    42
115911    39
117627    39
115888    38
123287    38
122277    33
134104    33
142385    31
119439    30
120261    30
126781    30
131812    29
133542    29
130654    27
132028    27
115714    25
120648    25
131820    25
143890    25
132727    24
117624    23
118505    23
120576    23
123963    23
131476    23
117721    22
118392    22
119703    22
120701    22
Name: customer_tweets, dtype: int64

In [16]:
candidate_analysis = pd.concat(
    [
        company_activity,
        customer_activity
    ],
    axis=1
).fillna(0)

candidate_analysis["total_activity"] = (
    candidate_analysis["company_replies"]
    + candidate_analysis["customer_tweets"]
)

candidate_analysis = candidate_analysis.sort_values(
    "company_replies",
    ascending=False
)

candidate_analysis.head(30)

,company_replies,customer_tweets,total_activity
author_id,,,
AmazonHelp,6924.0,0.0,6924.0
AppleSupport,3106.0,0.0,3106.0
Uber_Support,2102.0,0.0,2102.0
Tesco,1450.0,0.0,1450.0
SpotifyCares,1356.0,0.0,1356.0
British_Airways,1156.0,0.0,1156.0
comcastcares,1092.0,0.0,1092.0
sainsburys,1081.0,0.0,1081.0
Delta,1039.0,0.0,1039.0


In [17]:
candidate_analysis = pd.concat(
    [
        company_activity,
        customer_activity
    ],
    axis=1
).fillna(0)

candidate_analysis["total_activity"] = (
    candidate_analysis["company_replies"]
    + candidate_analysis["customer_tweets"]
)

candidate_analysis = candidate_analysis.sort_values(
    "company_replies",
    ascending=False
)

candidate_analysis.head(30)

,company_replies,customer_tweets,total_activity
author_id,,,
AmazonHelp,6924.0,0.0,6924.0
AppleSupport,3106.0,0.0,3106.0
Uber_Support,2102.0,0.0,2102.0
Tesco,1450.0,0.0,1450.0
SpotifyCares,1356.0,0.0,1356.0
British_Airways,1156.0,0.0,1156.0
comcastcares,1092.0,0.0,1092.0
sainsburys,1081.0,0.0,1081.0
Delta,1039.0,0.0,1039.0


In [18]:
# Investigate 

brand_name = "AmazonHelp"

amazon_tweets = df_sample[
    df_sample["author_id"] == brand_name
].copy()

print("AmazonHelp tweets:", len(amazon_tweets))

print("\nInbound distribution:")
print(amazon_tweets["inbound"].value_counts())

print("\nSample AmazonHelp replies:")
print(
    amazon_tweets[
        ["tweet_id", "text", "in_response_to_tweet_id", "response_tweet_id"]
    ].head(10).to_string(index=False)
)

AmazonHelp tweets: 6924

Inbound distribution:
inbound
False    6924
Name: count, dtype: int64

Sample AmazonHelp replies:
 tweet_id                                                                                                                                                      text  in_response_to_tweet_id response_tweet_id
      269                            @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET                    272.0           270,271
      273                                                                                     @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET                    271.0               274
      275                                                                                                    @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET                    274.0               NaN
      324 @115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止

In [19]:


amazon_reply_ids = set(
    amazon_tweets["in_response_to_tweet_id"]
    .dropna()
    .astype(str)
)

customer_messages = df_sample[
    df_sample["tweet_id"].astype(str).isin(amazon_reply_ids)
].copy()

print("Customer messages with AmazonHelp replies:",
      len(customer_messages))

customer_messages[
    ["tweet_id", "text", "in_response_to_tweet_id"]
].head(20)

Customer messages with AmazonHelp replies: 0


,tweet_id,text,in_response_to_tweet_id


In [20]:
conversation_examples = amazon_tweets.merge(
    customer_messages[
        ["tweet_id", "text"]
    ],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    how="inner",
    suffixes=("_brand", "_customer")
)

print("Matched conversations:",
      len(conversation_examples))

conversation_examples[
    ["text_customer", "text_brand"]
].head(20)

Matched conversations: 0


,text_customer,text_brand


In [21]:
coverage = len(conversation_examples) / len(amazon_tweets)

print(f"AmazonHelp reply-to-customer coverage: {coverage:.2%}")

AmazonHelp reply-to-customer coverage: 0.00%


In [22]:
for i, row in conversation_examples.head(5).iterrows():
    print("=" * 80)
    print("CUSTOMER:")
    print(row["text_customer"])
    print("\nAMAZONHELP:")
    print(row["text_brand"])

In [23]:
def analyze_brand(brand_name, df):
    brand_tweets = df[
        df["author_id"] == brand_name
    ].copy()

    reply_ids = set(
        brand_tweets["in_response_to_tweet_id"]
        .dropna()
        .astype(str)
    )

    customer_msgs = df[
        df["tweet_id"].astype(str).isin(reply_ids)
    ]

    matched = brand_tweets.merge(
        customer_msgs[["tweet_id", "text"]],
        left_on="in_response_to_tweet_id",
        right_on="tweet_id",
        how="inner",
        suffixes=("_brand", "_customer")
    )

    return {
        "brand": brand_name,
        "brand_replies": len(brand_tweets),
        "matched_customer_messages": len(matched),
        "coverage": (
            len(matched) / len(brand_tweets)
            if len(brand_tweets) > 0 else 0
        )
    }


brands_to_compare = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "British_Airways"
]

comparison = pd.DataFrame(
    [analyze_brand(b, df_sample) for b in brands_to_compare]
)

comparison.sort_values(
    "matched_customer_messages",
    ascending=False
)

,brand,brand_replies,matched_customer_messages,coverage
0,AmazonHelp,6924,0,0.0
1,AppleSupport,3106,0,0.0
2,Uber_Support,2102,0,0.0
3,SpotifyCares,1356,0,0.0
4,British_Airways,1156,0,0.0


In [24]:
comparison.sort_values(
    "matched_customer_messages",
    ascending=False
)

,brand,brand_replies,matched_customer_messages,coverage
0,AmazonHelp,6924,0,0.0
1,AppleSupport,3106,0,0.0
2,Uber_Support,2102,0,0.0
3,SpotifyCares,1356,0,0.0
4,British_Airways,1156,0,0.0


In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "twcs.csv"

print("Dataset path:")
print(DATA_PATH)

# Load only 100,000 rows for exploration
df_sample = pd.read_csv(
    DATA_PATH,
    nrows=100_000
)

print("\nDataset loaded!")
print("Shape:", df_sample.shape)

# Normalize ID columns
def normalize_id(series):
    return pd.to_numeric(
        series,
        errors="coerce"
    ).astype("Int64")


df_sample["tweet_id_num"] = normalize_id(
    df_sample["tweet_id"]
)

df_sample["response_tweet_id_num"] = normalize_id(
    df_sample["response_tweet_id"]
)

df_sample["in_response_to_tweet_id_num"] = normalize_id(
    df_sample["in_response_to_tweet_id"]
)

print("\nID normalization complete!")

Dataset path:
c:\Users\arnav\OneDrive\Desktop\hiver-ai-support-agent\data\raw\twcs.csv

Dataset loaded!
Shape: (100000, 7)

ID normalization complete!


In [27]:
from collections import Counter

company_counts = Counter()
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000
):
    total_rows += len(chunk)

    company_rows = chunk[
        chunk["inbound"] == False
    ]

    company_counts.update(
        company_rows["author_id"]
        .dropna()
        .astype(str)
    )

print(f"Total rows scanned: {total_rows:,}")
print(f"Company accounts found: {len(company_counts):,}")

Total rows scanned: 2,811,774
Company accounts found: 108


In [28]:
top_brands = pd.DataFrame(
    company_counts.most_common(30),
    columns=["brand", "company_replies"]
)

top_brands

,brand,company_replies
0,AmazonHelp,169840
1,AppleSupport,106860
2,Uber_Support,56270
3,SpotifyCares,43265
4,Delta,42253
5,Tesco,38573
6,AmericanAir,36764
7,TMobileHelp,34317
8,comcastcares,33031
9,British_Airways,29361


In [29]:
BRAND = "AmazonHelp"

brand_chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000
):
    matches = chunk[
        chunk["author_id"].astype(str) == BRAND
    ]

    if not matches.empty:
        brand_chunks.append(matches)

amazon_full = pd.concat(
    brand_chunks,
    ignore_index=True
)

print("AmazonHelp tweets:", len(amazon_full))

AmazonHelp tweets: 169840


In [30]:
amazon_full[
    [
        "tweet_id",
        "author_id",
        "inbound",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ]
].head(20)

,tweet_id,author_id,inbound,text,response_tweet_id,in_response_to_tweet_id
0,269,AmazonHelp,False,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
1,273,AmazonHelp,False,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
2,275,AmazonHelp,False,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
3,324,AmazonHelp,False,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
4,615,AmazonHelp,False,@115820 I'm sorry we've let you down! Without ...,616,617.0
5,618,AmazonHelp,False,@115820 We'd like to take a further look into ...,619,616.0
6,620,AmazonHelp,False,@115822 I am unable to affect your account via...,NaN,621.0
7,622,AmazonHelp,False,"@115824 Hi, wir erhalten die Filme/Serien so v...",623,624.0
8,625,AmazonHelp,False,@115824 Wir haben zu danken. Schönen Abend noc...,NaN,623.0
9,626,AmazonHelp,False,@115826 I'm sorry for the wait. You'll receive...,627,628.0


In [31]:
response_ids = set()

for value in amazon_full["response_tweet_id"].dropna():
    for tweet_id in str(value).split(","):
        tweet_id = tweet_id.strip()

        if tweet_id:
            response_ids.add(tweet_id)

print("Linked response tweet IDs:", len(response_ids))

Linked response tweet IDs: 100785


In [32]:
customer_chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000
):
    chunk_ids = chunk["tweet_id"].astype(str)

    matches = chunk[
        chunk_ids.isin(response_ids)
    ]

    if not matches.empty:
        customer_chunks.append(matches)

linked_tweets = pd.concat(
    customer_chunks,
    ignore_index=True
) if customer_chunks else pd.DataFrame()

print(
    "Linked tweets found:",
    len(linked_tweets)
)

Linked tweets found: 100513


In [33]:
amazon_pairs = amazon_full.merge(
    linked_tweets[
        ["tweet_id", "text", "author_id", "inbound"]
    ],
    left_on=amazon_full["response_tweet_id"]
        .astype(str)
        .str.split(",")
        .str[0],
    right_on=linked_tweets["tweet_id"].astype(str),
    how="inner",
    suffixes=("_brand", "_customer")
)

In [34]:
response_mapping = []

for _, row in amazon_full.iterrows():
    response_value = row["response_tweet_id"]

    if pd.isna(response_value):
        continue

    for response_id in str(response_value).split(","):
        response_id = response_id.strip()

        if response_id:
            response_mapping.append({
                "brand_tweet_id": str(row["tweet_id"]),
                "customer_tweet_id": response_id,
                "brand_text": row["text"]
            })

response_mapping = pd.DataFrame(response_mapping)

print(
    "Brand → linked tweet relationships:",
    len(response_mapping)
)

Brand → linked tweet relationships: 100785


In [35]:
amazon_pairs = response_mapping.merge(
    linked_tweets[
        ["tweet_id", "text", "author_id", "inbound"]
    ],
    left_on="customer_tweet_id",
    right_on=linked_tweets["tweet_id"].astype(str),
    how="inner"
)

amazon_pairs = amazon_pairs.rename(
    columns={
        "text": "customer_text"
    }
)

print("Matched AmazonHelp conversations:", len(amazon_pairs))

Matched AmazonHelp conversations: 100513


In [36]:
for _, row in amazon_pairs.head(10).iterrows():

    print("=" * 90)

    print("CUSTOMER:")
    print(row["customer_text"])

    print("\nAMAZONHELP:")
    print(row["brand_text"])

CUSTOMER:
@AmazonHelp ありがとうございます。
今、電話で主人が対応していただいてます。

AMAZONHELP:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
CUSTOMER:
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。

AMAZONHELP:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
CUSTOMER:
@AmazonHelp こちらこそありがとうございました。

AMAZONHELP:
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
CUSTOMER:
@AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day

AMAZONHELP:
@115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN
CUSTOMER:
@AmazonHelp I frankly don't have the patience for another chat with your "customer service" people today.

AMAZONHELP:
@115820 We'd like to take a furthe

In [37]:
from collections import Counter
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "twcs.csv"

print("Dataset:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

Dataset: c:\Users\arnav\OneDrive\Desktop\hiver-ai-support-agent\data\raw\twcs.csv
Exists: True


In [38]:
company_counts = Counter()
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000
):
    total_rows += len(chunk)

    company_rows = chunk[chunk["inbound"] == False]

    company_counts.update(
        company_rows["author_id"].dropna().astype(str)
    )

print(f"Total rows scanned: {total_rows:,}")
print(f"Company accounts found: {len(company_counts):,}")

Total rows scanned: 2,811,774
Company accounts found: 108


In [ ]:
top_brands = pd.DataFrame(
    company_counts.most_common(30),
    columns=["brand", "company_replies"]
)

top_brands

,brand,company_replies
0,AmazonHelp,169840
1,AppleSupport,106860
2,Uber_Support,56270
3,SpotifyCares,43265
4,Delta,42253
5,Tesco,38573
6,AmericanAir,36764
7,TMobileHelp,34317
8,comcastcares,33031
9,British_Airways,29361


In [40]:
BRAND = "AmazonHelp"

brand_tweets = df_sample[
    df_sample["author_id"].astype(str) == BRAND
].copy()

print("Brand tweets:", len(brand_tweets))

Brand tweets: 6924


In [41]:
customer_tweets = df_sample[
    df_sample["inbound"] == True
].copy()

# Convert brand tweet IDs to strings
brand_ids = set(
    brand_tweets["tweet_id"].astype(str)
)

# Find customer tweets whose response_tweet_id
# points to an AmazonHelp tweet
def contains_brand_reply(value):
    if pd.isna(value):
        return False

    ids = [
        x.strip()
        for x in str(value).split(",")
    ]

    return any(x in brand_ids for x in ids)


linked_customers = customer_tweets[
    customer_tweets["response_tweet_id"]
    .apply(contains_brand_reply)
].copy()

print(
    "Customer tweets linked to AmazonHelp:",
    len(linked_customers)
)

Customer tweets linked to AmazonHelp: 6516


In [42]:
# Create one row for every Customer → AmazonHelp relationship

conversation_rows = []

for _, customer in linked_customers.iterrows():

    response_ids = [
        x.strip()
        for x in str(customer["response_tweet_id"]).split(",")
        if x.strip()
    ]

    for brand_tweet_id in response_ids:

        if brand_tweet_id in brand_ids:

            brand_row = brand_tweets[
                brand_tweets["tweet_id"].astype(str)
                == brand_tweet_id
            ]

            if not brand_row.empty:

                brand_row = brand_row.iloc[0]

                conversation_rows.append({
                    "customer_tweet_id": customer["tweet_id"],
                    "customer_text": customer["text"],
                    "brand_tweet_id": brand_row["tweet_id"],
                    "brand_text": brand_row["text"]
                })

amazon_conversations = pd.DataFrame(
    conversation_rows
)

print(
    "Customer → AmazonHelp conversations:",
    len(amazon_conversations)
)

Customer → AmazonHelp conversations: 6893


In [43]:
amazon_conversations[
    [
        "customer_text",
        "brand_text"
    ]
].head(10)

,customer_text,brand_text
0,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
1,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
2,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
3,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
5,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
6,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
7,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
8,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
9,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM


In [44]:
for _, row in amazon_conversations.head(5).iterrows():

    print("=" * 100)

    print("CUSTOMER:")
    print(row["customer_text"])

    print("\nAMAZONHELP:")
    print(row["brand_text"])

    print()

CUSTOMER:
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。

AMAZONHELP:
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET

CUSTOMER:
@AmazonHelp こちらこそありがとうございました。

AMAZONHELP:
@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET

CUSTOMER:
amazonのfireTVstickが見れない😢

AMAZONHELP:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET

CUSTOMER:
amazonプライムビデオ、再生エラーが多いです

AMAZONHELP:
@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善する場合がございますので、お試しください。改善しない場合は、状況を確認しご案内させていただきますのでこちらからカスタマーサービスまでご連絡ください。https://t.co/NtNAX2Qh2u ET

CUSTOMER:
@AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day

AMAZONHELP:
@115820 We'd like to take a further look into this with you! Please reach us by phone or chat here: https://t.co/hApLpMlfHN ^AG



In [45]:
# Keep only useful conversation fields
amazon_data = amazon_conversations[
    [
        "customer_tweet_id",
        "customer_text",
        "brand_tweet_id",
        "brand_text"
    ]
].copy()

# Remove missing messages
amazon_data = amazon_data.dropna(
    subset=["customer_text", "brand_text"]
)

# Convert to string
amazon_data["customer_text"] = (
    amazon_data["customer_text"]
    .astype(str)
    .str.strip()
)

amazon_data["brand_text"] = (
    amazon_data["brand_text"]
    .astype(str)
    .str.strip()
)

# Remove empty messages
amazon_data = amazon_data[
    (amazon_data["customer_text"].str.len() > 5)
]

print("Conversation pairs:", len(amazon_data))

Conversation pairs: 6893


In [46]:
amazon_data[
    ["customer_text", "brand_text"]
].head(20)

,customer_text,brand_text
0,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
1,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
2,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
3,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
5,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
6,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
7,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
8,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
9,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM


In [47]:
generic_patterns = [
    "please help",
    "dm me",
    "sent you a dm",
    "direct message",
    "private message",
    "thank you",
    "thanks",
    "hello",
    "hi"
]

generic_mask = amazon_data["customer_text"].str.lower().apply(
    lambda text: any(pattern in text for pattern in generic_patterns)
)

print("Potentially generic messages:", generic_mask.sum())

Potentially generic messages: 2194


In [48]:
amazon_data["customer_length"] = (
    amazon_data["customer_text"]
    .str.len()
)

amazon_data["customer_length"].describe()

count    6893.000000
mean      122.249238
std        65.259938
min         7.000000
25%        72.000000
50%       119.000000
75%       152.000000
max       365.000000
Name: customer_length, dtype: float64

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(
    amazon_data["customer_text"]
)

print("TF-IDF matrix shape:", X.shape)

TF-IDF matrix shape: (6893, 1511)


In [50]:
import numpy as np

terms = np.array(
    vectorizer.get_feature_names_out()
)

scores = np.asarray(
    X.mean(axis=0)
).ravel()

top_indices = scores.argsort()[::-1][:50]

top_terms = pd.DataFrame({
    "term": terms[top_indices],
    "tfidf_score": scores[top_indices]
})

top_terms

,term,tfidf_score
0,amazonhelp,0.090577
1,https,0.038680
2,amazon,0.027972
3,115850,0.020952
4,115821,0.020828
5,order,0.019897
6,delivery,0.018937
7,prime,0.016475
8,115830,0.013491
9,que,0.012845


In [51]:
from sklearn.cluster import KMeans

n_clusters = 12

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

amazon_data["cluster"] = kmeans.fit_predict(X)

print("Clusters created:", n_clusters)

Clusters created: 12


In [52]:
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_id in range(n_clusters):

    top_words = terms[
        order_centroids[cluster_id, :10]
    ]

    print(
        f"\nCluster {cluster_id}:"
    )
    print(", ".join(top_words))


Cluster 0:
115821, delivery, day, 115830, prime, https, shipping, package, days, ordered

Cluster 1:
amazonhelp, 115851, worth, 115830, 115830 amazonhelp, yo, 00, ça, 115825, 115821 prime

Cluster 2:
ich, nicht, die, 116316, und, das, ist, es, der, für

Cluster 3:
https, amazonhelp https, amazonhelp, amazon, 119625, check, 115830, 120533, 115821, 11

Cluster 4:
delivered, amazonhelp, delivery, package, today, delivered today, order, says, item, day

Cluster 5:
que, el, en, la, lo, 116875, mi, por, se, 116928

Cluster 6:
amazonhelp, https, just, ve, yes, help, thanks, email, time, amazonhelp yes

Cluster 7:
amazon, amazonhelp, amazonhelp amazon, amazon prime, prime, https, india, amazon logistics, logistics, 115821

Cluster 8:
115850, https, amazon, amazonhelp 115850, 115850 amazonhelp, 115851, amazonhelp, product, refund, 115821

Cluster 9:
order, amazonhelp, amazonhelp order, 115850, delivery, amazon, https, placed, 115850 order, cancelled

Cluster 10:
le, je, est, pas, et, 120533, a

In [53]:
for cluster_id in range(n_clusters):

    print("\n" + "=" * 100)
    print(f"CLUSTER {cluster_id}")

    examples = amazon_data[
        amazon_data["cluster"] == cluster_id
    ]["customer_text"].head(5)

    for example in examples:
        print("-", example)


CLUSTER 0
- @115828 How about you guys figure out my Xbox One X project Scorpio edition first. No expected delivery or shipping date and it’s only a week away
- @115830 my package was ‘accidentally’ opened.. 4 items missing worth £97.
You need better delivery drivers!! https://t.co/f6SaVBSMqM
- Bought an @115821 Echo Show and it won’t recognize a single @AmazonHelp account in our household. WTF, guys?
- In response to your @115830 packing video, this packaging was for a 2ft washing line pole @115837 https://t.co/X21SQHgC0K
- @115821, it’d be nice if the book I waited 4 months for wasn’t damaged inside of an undented box. #twinpeaks #twin… https://t.co/Lac4K7iQzJ https://t.co/JroqJKrH9Q

CLUSTER 1
- @AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。
- @AmazonHelp こちらこそありがとうございました。
- @AmazonHelp I have. No luck.
- @AmazonHelp ご丁寧にありがとうございます！トライしてみて改善されなければCSへご連絡します。返信いただけると思っていなかったので本当に嬉しいです！
- @AmazonHelp All of them

CLUSTER 2
- @AmazonHelp Okay, danke für die Info
- @115

In [54]:
INTENTS = {
    "refund_status": "Customer asking about a pending or missing refund.",
    "delivery_issue": "Customer reporting a delayed, missing, or late delivery.",
    "payment_issue": "Customer reporting a charge, payment, or billing problem.",
    "account_issue": "Customer having trouble accessing or managing their account.",
    # ...
}

In [55]:
for cluster_id in range(n_clusters):
    ...

In [56]:
# Show actual examples from every discovered cluster

for cluster_id in sorted(amazon_data["cluster"].unique()):

    print("\n" + "=" * 100)
    print(f"CLUSTER {cluster_id}")

    examples = amazon_data[
        amazon_data["cluster"] == cluster_id
    ]["customer_text"].head(8)

    for i, example in enumerate(examples, 1):
        print(f"{i}. {example}")


CLUSTER 0
1. @115828 How about you guys figure out my Xbox One X project Scorpio edition first. No expected delivery or shipping date and it’s only a week away
2. @115830 my package was ‘accidentally’ opened.. 4 items missing worth £97.
You need better delivery drivers!! https://t.co/f6SaVBSMqM
3. Bought an @115821 Echo Show and it won’t recognize a single @AmazonHelp account in our household. WTF, guys?
4. In response to your @115830 packing video, this packaging was for a 2ft washing line pole @115837 https://t.co/X21SQHgC0K
5. @115821, it’d be nice if the book I waited 4 months for wasn’t damaged inside of an undented box. #twinpeaks #twin… https://t.co/Lac4K7iQzJ https://t.co/JroqJKrH9Q
6. @AmazonHelp delivery I paid for today,didn’t arrive.why not?i paid enough for it.where is it??I’m unhappy.refund the delivery charge
7. Anna Inspired in idea lab at school to be @115821 package being shipped to Narnia! "Amazon can go anywhere" according to Anna. https://t.co/TyvKhuu7su
8. @11584

In [57]:
for cluster_id in sorted(amazon_data["cluster"].unique()):

    print("\n" + "=" * 100)
    print(f"CLUSTER {cluster_id}")

    examples = amazon_data[
        amazon_data["cluster"] == cluster_id
    ]["customer_text"].head(8)

    for i, example in enumerate(examples, 1):
        print(f"{i}. {example}")


CLUSTER 0
1. @115828 How about you guys figure out my Xbox One X project Scorpio edition first. No expected delivery or shipping date and it’s only a week away
2. @115830 my package was ‘accidentally’ opened.. 4 items missing worth £97.
You need better delivery drivers!! https://t.co/f6SaVBSMqM
3. Bought an @115821 Echo Show and it won’t recognize a single @AmazonHelp account in our household. WTF, guys?
4. In response to your @115830 packing video, this packaging was for a 2ft washing line pole @115837 https://t.co/X21SQHgC0K
5. @115821, it’d be nice if the book I waited 4 months for wasn’t damaged inside of an undented box. #twinpeaks #twin… https://t.co/Lac4K7iQzJ https://t.co/JroqJKrH9Q
6. @AmazonHelp delivery I paid for today,didn’t arrive.why not?i paid enough for it.where is it??I’m unhappy.refund the delivery charge
7. Anna Inspired in idea lab at school to be @115821 package being shipped to Narnia! "Amazon can go anywhere" according to Anna. https://t.co/TyvKhuu7su
8. @11584